# Delta Lake MERGE Implementation — Incremental Data Processing

**Objective:** Perform incremental data processing using Delta Lake.

**Dataset:** Synthetic customer dimension data
(`customer_master.csv` = existing/base customer records, `customer_incremental.csv` =
new & updated customer records arriving as a feed).

**Steps covered in this notebook:**
1. Load dataset into a Delta table
2. Basic cleaning (handle nulls, remove duplicates)
3. Create/inspect the incremental dataset (new + changed records)
4. Apply `MERGE` to update existing records and insert new ones
   - 4a. SCD Type 1 (overwrite in place — no history kept)
   - 4b. SCD Type 2 (preserve history — old versions kept with `is_current` / `end_date`)
5. Validate results (row counts, duplicate checks, spot checks)
6. Display final dataset + summary

**Engine:** [`delta-rs`](https://github.com/delta-io/delta-rs) via the `deltalake` Python
package — a real Delta Lake implementation (Rust core, Python bindings) that runs anywhere
with plain Python, no Spark cluster or JVM required. Every `MERGE` call below runs against
an actual Delta table on disk and produces a real, versioned transaction log. The same
semantics apply 1:1 to PySpark (`DeltaTable.forPath(spark, path).merge(...)`).

In [1]:
import os
import shutil
import pandas as pd
from deltalake import DeltaTable, write_deltalake

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

BASE_DIR = os.path.abspath(".")
DATA_DIR = os.path.join(BASE_DIR, "data")
DELTA_DIR = os.path.join(BASE_DIR, "delta")

# Start clean so the notebook is idempotent/re-runnable
if os.path.exists(DELTA_DIR):
    shutil.rmtree(DELTA_DIR)
os.makedirs(DELTA_DIR, exist_ok=True)

BRONZE_PATH = os.path.join(DELTA_DIR, "customers_bronze")    # raw, as-loaded
SILVER_SCD1_PATH = os.path.join(DELTA_DIR, "customers_scd1")  # cleaned + SCD1 merges
SILVER_SCD2_PATH = os.path.join(DELTA_DIR, "customers_scd2")  # cleaned + SCD2 merges

print("Data dir :", DATA_DIR)
print("Delta dir:", DELTA_DIR)

Data dir : /home/claude/work/assignment/data
Delta dir: /home/claude/work/assignment/delta


## Step 1 — Load dataset into a Delta table

We read the base customer file (`customer_master.csv`) and write it into a Delta table
(the "bronze" layer). This is our starting point before any incremental changes arrive.

In [2]:
master_df = pd.read_csv(os.path.join(DATA_DIR, "customer_master.csv"))

print(f"Rows read from customer_master.csv: {len(master_df)}")
master_df.head(10)

Rows read from customer_master.csv: 153


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
0,C-1000,Mona Ortiz,Consumer,United States,Seattle,Washington,80239.0,West,2024-01-01
1,C-1001,Sam Ortiz,Home Office,United States,Columbus,Ohio,38140.0,Central,2024-01-01
2,C-1002,Noah Clark,Consumer,United States,Chicago,Illinois,41544.0,Central,2024-01-01
3,C-1003,Noah Ortiz,Home Office,United States,Houston,Texas,26226.0,Central,2024-01-01
4,C-1004,Sam Okafor,Corporate,United States,New York City,New York,16499.0,East,2024-01-01
5,C-1005,Rosa Popov,Corporate,United States,New York City,New York,64937.0,East,2024-01-01
6,C-1006,Sam Farouk,Home Office,United States,New York City,New York,99391.0,East,2024-01-01
7,C-1007,Sam Okafor,Home Office,United States,New York City,New York,34624.0,East,2024-01-01
8,C-1008,Rosa Reyes,Home Office,United States,Oakland,California,17812.0,West,2024-01-01
9,C-1009,Vera Diaz,Corporate,United States,Newark,New Jersey,51175.0,East,2024-01-01


In [3]:
write_deltalake(BRONZE_PATH, master_df, mode="overwrite")

dt_bronze = DeltaTable(BRONZE_PATH)
print("Delta table created at:", BRONZE_PATH)
print("Delta table version   :", dt_bronze.version())
print("Row count in Delta table:", len(dt_bronze.to_pandas()))

Delta table created at: /home/claude/work/assignment/delta/customers_bronze
Delta table version   : 0
Row count in Delta table: 153


## Step 2 — Basic cleaning (handle nulls, remove duplicates)

We check for exact duplicate rows, duplicate `customer_id`s (should be a unique key), and
nulls in important columns — then clean the data before it becomes our trusted "silver" table.

In [4]:
bronze_df = dt_bronze.to_pandas()

print("=== BEFORE CLEANING ===")
print("Total rows                 :", len(bronze_df))
print("Fully duplicated rows      :", bronze_df.duplicated().sum())
print("Duplicate customer_id rows :", bronze_df.duplicated(subset=['customer_id']).sum())
print("\nNull counts per column:")
print(bronze_df.isnull().sum())

=== BEFORE CLEANING ===
Total rows                 : 153
Fully duplicated rows      : 3
Duplicate customer_id rows : 3

Null counts per column:
customer_id      0
customer_name    0
segment          4
country          0
city             0
state            0
postal_code      5
region           0
last_updated     0
dtype: int64


In [5]:
cleaned_df = bronze_df.copy()

# 1) Remove fully duplicated rows
cleaned_df = cleaned_df.drop_duplicates()

# 2) Remove duplicate customer_id rows, keeping the most recently updated record
cleaned_df = (
    cleaned_df.sort_values('last_updated')
              .drop_duplicates(subset=['customer_id'], keep='last')
)

# 3) Handle nulls
cleaned_df['segment'] = cleaned_df['segment'].fillna('Unknown')
cleaned_df['postal_code'] = cleaned_df['postal_code'].fillna(0).astype(int).astype(str)

cleaned_df = cleaned_df.reset_index(drop=True)

print("=== AFTER CLEANING ===")
print("Total rows                 :", len(cleaned_df))
print("Fully duplicated rows      :", cleaned_df.duplicated().sum())
print("Duplicate customer_id rows :", cleaned_df.duplicated(subset=['customer_id']).sum())
print("Remaining nulls            :", int(cleaned_df.isnull().sum().sum()))
cleaned_df.head(10)

=== AFTER CLEANING ===
Total rows                 : 150
Fully duplicated rows      : 0
Duplicate customer_id rows : 0
Remaining nulls            : 0


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
0,C-1000,Mona Ortiz,Consumer,United States,Seattle,Washington,80239,West,2024-01-01
1,C-1001,Sam Ortiz,Home Office,United States,Columbus,Ohio,38140,Central,2024-01-01
2,C-1002,Noah Clark,Consumer,United States,Chicago,Illinois,41544,Central,2024-01-01
3,C-1003,Noah Ortiz,Home Office,United States,Houston,Texas,26226,Central,2024-01-01
4,C-1004,Sam Okafor,Corporate,United States,New York City,New York,16499,East,2024-01-01
5,C-1005,Rosa Popov,Corporate,United States,New York City,New York,64937,East,2024-01-01
6,C-1006,Sam Farouk,Home Office,United States,New York City,New York,99391,East,2024-01-01
7,C-1007,Sam Okafor,Home Office,United States,New York City,New York,34624,East,2024-01-01
8,C-1008,Rosa Reyes,Home Office,United States,Oakland,California,17812,West,2024-01-01
9,C-1009,Vera Diaz,Corporate,United States,Newark,New Jersey,51175,East,2024-01-01


In [6]:
# Persist the cleaned data as our two starting Delta tables
write_deltalake(SILVER_SCD1_PATH, cleaned_df, mode="overwrite")

scd2_seed_df = cleaned_df.copy()
scd2_seed_df['effective_date'] = scd2_seed_df['last_updated']
scd2_seed_df['end_date'] = ''
scd2_seed_df['is_current'] = True
write_deltalake(SILVER_SCD2_PATH, scd2_seed_df, mode="overwrite")

print("Cleaned rows written to SCD1 table:", len(DeltaTable(SILVER_SCD1_PATH).to_pandas()))
print("Cleaned rows written to SCD2 table:", len(DeltaTable(SILVER_SCD2_PATH).to_pandas()))

Cleaned rows written to SCD1 table: 150
Cleaned rows written to SCD2 table: 150


## Step 3 — Create/inspect the incremental dataset

`customer_incremental.csv` simulates a new batch/feed arriving: some rows are **updates**
to existing customers, and some are **brand-new customers** (`customer_id` starting with `NC-`).

In [7]:
incremental_df = pd.read_csv(os.path.join(DATA_DIR, "customer_incremental.csv"))

existing_ids = set(cleaned_df['customer_id'])
incremental_df['change_type'] = incremental_df['customer_id'].apply(
    lambda cid: 'NEW' if cid not in existing_ids else 'UPDATE'
)

print(f"Rows in incremental feed        : {len(incremental_df)}")
print(f"  -> UPDATE (existing customer) : {(incremental_df['change_type']=='UPDATE').sum()}")
print(f"  -> NEW (brand-new customer)   : {(incremental_df['change_type']=='NEW').sum()}")
incremental_df.head(10)

Rows in incremental feed        : 28
  -> UPDATE (existing customer) : 20
  -> NEW (brand-new customer)   : 8


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated,change_type
0,C-1080,Ben Novak,Home Office,United States,Newark,New Jersey,65123,East,2024-06-15,UPDATE
1,C-1062,Grace Nguyen,Corporate,United States,Reno,Nevada,76175,West,2024-06-15,UPDATE
2,C-1121,Priya Liu,Corporate,United States,Green Bay,Wisconsin,35419,Central,2024-06-15,UPDATE
3,C-1134,Hiro Silva,Corporate,United States,Newark,New Jersey,24287,East,2024-06-15,UPDATE
4,C-1060,Carlos Tanaka,Corporate,United States,Atlanta,Georgia,64660,South,2024-06-15,UPDATE
5,C-1140,Elena Ahmadi,Consumer,United States,Houston,Texas,37911,Central,2024-06-15,UPDATE
6,C-1063,Elena Clark,Consumer,United States,Houston,Texas,17882,Central,2024-06-15,UPDATE
7,C-1007,Olga Novak,Home Office,United States,Newark,New Jersey,24838,East,2024-06-15,UPDATE
8,C-1105,Karin Liu,Consumer,United States,Green Bay,Wisconsin,95520,Central,2024-06-15,UPDATE
9,C-1078,Jamal Ahmadi,Corporate,United States,Tampa,Florida,53476,South,2024-06-15,UPDATE


## Step 4a — Apply MERGE: SCD Type 1 (overwrite in place)

SCD Type 1 simply **overwrites** the attributes of matching customers with the latest
values, and **inserts** brand-new customers. No history is kept.

In [8]:
merge_source = incremental_df.drop(columns=['change_type'])

dt_scd1 = DeltaTable(SILVER_SCD1_PATH)

(
    dt_scd1.merge(
        source=merge_source,
        predicate="target.customer_id = source.customer_id",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)

print("SCD1 MERGE complete. New table version:", dt_scd1.version())
scd1_result = dt_scd1.to_pandas().sort_values('customer_id').reset_index(drop=True)
print("Total rows after merge:", len(scd1_result))
scd1_result.head(10)

SCD1 MERGE complete. New table version: 1
Total rows after merge: 158


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
0,C-1000,Mona Ortiz,Consumer,United States,Seattle,Washington,80239,West,2024-01-01
1,C-1001,Sam Ortiz,Home Office,United States,Columbus,Ohio,38140,Central,2024-01-01
2,C-1002,Noah Clark,Consumer,United States,Chicago,Illinois,41544,Central,2024-01-01
3,C-1003,Noah Ortiz,Home Office,United States,Houston,Texas,26226,Central,2024-01-01
4,C-1004,Sam Okafor,Corporate,United States,New York City,New York,16499,East,2024-01-01
5,C-1005,Noah Kim,Home Office,United States,Columbus,Ohio,37184,Central,2024-06-15
6,C-1006,Sam Farouk,Home Office,United States,New York City,New York,99391,East,2024-01-01
7,C-1007,Olga Novak,Home Office,United States,Newark,New Jersey,24838,East,2024-06-15
8,C-1008,Rosa Reyes,Home Office,United States,Oakland,California,17812,West,2024-01-01
9,C-1009,Vera Diaz,Corporate,United States,Newark,New Jersey,51175,East,2024-01-01


In [9]:
updated_id = incremental_df.loc[incremental_df['change_type']=='UPDATE', 'customer_id'].iloc[0]
print(f"Customer {updated_id} — SCD1 table (only ONE row, latest values, no history kept):")
scd1_result[scd1_result['customer_id']==updated_id]

Customer C-1080 — SCD1 table (only ONE row, latest values, no history kept):


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
80,C-1080,Ben Novak,Home Office,United States,Newark,New Jersey,65123,East,2024-06-15


## Step 4b — Apply MERGE: SCD Type 2 (preserve history)

SCD Type 2 keeps every version of a customer's record. When an update arrives we:
1. `WHEN MATCHED AND target.is_current = true THEN UPDATE` the old row's
   `is_current -> false` and stamp an `end_date`
2. `WHEN NOT MATCHED THEN INSERT` the new version as a fresh, current row

This requires **two merge passes**.

In [10]:
dt_scd2 = DeltaTable(SILVER_SCD2_PATH)

close_source = merge_source[['customer_id', 'last_updated']].rename(columns={'last_updated': 'new_effective_date'})

(
    dt_scd2.merge(
        source=close_source,
        predicate="target.customer_id = source.customer_id AND target.is_current = true",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update(
        updates={
            "is_current": "false",
            "end_date": "source.new_effective_date",
        }
    )
    .execute()
)

print("Pass 1 (close old versions) complete. Table version:", dt_scd2.version())

Pass 1 (close old versions) complete. Table version: 1


In [11]:
insert_source = merge_source.copy()
insert_source['effective_date'] = insert_source['last_updated']
insert_source['end_date'] = ''
insert_source['is_current'] = True

dt_scd2 = DeltaTable(SILVER_SCD2_PATH)
(
    dt_scd2.merge(
        source=insert_source,
        predicate="target.customer_id = source.customer_id AND target.effective_date = source.effective_date",
        source_alias="source",
        target_alias="target",
    )
    .when_not_matched_insert_all()
    .execute()
)

print("Pass 2 (insert new versions) complete. Final table version:", dt_scd2.version())
scd2_result = dt_scd2.to_pandas().sort_values(['customer_id', 'effective_date']).reset_index(drop=True)
print("Total rows after SCD2 merge (includes historical versions):", len(scd2_result))
scd2_result.head(10)

Pass 2 (insert new versions) complete. Final table version: 2
Total rows after SCD2 merge (includes historical versions): 178


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated,effective_date,end_date,is_current
0,C-1000,Mona Ortiz,Consumer,United States,Seattle,Washington,80239,West,2024-01-01,2024-01-01,,True
1,C-1001,Sam Ortiz,Home Office,United States,Columbus,Ohio,38140,Central,2024-01-01,2024-01-01,,True
2,C-1002,Noah Clark,Consumer,United States,Chicago,Illinois,41544,Central,2024-01-01,2024-01-01,,True
3,C-1003,Noah Ortiz,Home Office,United States,Houston,Texas,26226,Central,2024-01-01,2024-01-01,,True
4,C-1004,Sam Okafor,Corporate,United States,New York City,New York,16499,East,2024-01-01,2024-01-01,,True
5,C-1005,Rosa Popov,Corporate,United States,New York City,New York,64937,East,2024-01-01,2024-01-01,2024-06-15,False
6,C-1005,Noah Kim,Home Office,United States,Columbus,Ohio,37184,Central,2024-06-15,2024-06-15,,True
7,C-1006,Sam Farouk,Home Office,United States,New York City,New York,99391,East,2024-01-01,2024-01-01,,True
8,C-1007,Sam Okafor,Home Office,United States,New York City,New York,34624,East,2024-01-01,2024-01-01,2024-06-15,False
9,C-1007,Olga Novak,Home Office,United States,Newark,New Jersey,24838,East,2024-06-15,2024-06-15,,True


In [12]:
print(f"Customer {updated_id} — SCD2 table (history preserved):")
scd2_result[scd2_result['customer_id']==updated_id][
    ['customer_id','customer_name','city','state','segment','effective_date','end_date','is_current']
]

Customer C-1080 — SCD2 table (history preserved):


,customer_id,customer_name,city,state,segment,effective_date,end_date,is_current
91,C-1080,Ben Sullivan,Seattle,Washington,Home Office,2024-01-01,2024-06-15,False
92,C-1080,Ben Novak,Newark,New Jersey,Home Office,2024-06-15,,True


## Step 5 — Validate results

- Row counts make sense given (existing + new)
- No duplicate `customer_id` in SCD1
- No duplicate *current* `customer_id` in SCD2
- All new customers from the incremental feed are present

In [13]:
n_master_clean = len(cleaned_df)
n_new = (incremental_df['change_type']=='NEW').sum()
n_updates = (incremental_df['change_type']=='UPDATE').sum()

print("=== ROW COUNT VALIDATION ===")
print(f"Cleaned master rows        : {n_master_clean}")
print(f"Incremental NEW customers  : {n_new}")
print(f"Incremental UPDATEs        : {n_updates}")
print(f"Expected SCD1 row count    : {n_master_clean + n_new}  (updates overwrite, don't add rows)")
print(f"Actual SCD1 row count      : {len(scd1_result)}")
assert len(scd1_result) == n_master_clean + n_new, "SCD1 row count mismatch!"
print("PASS: SCD1 row count matches expectation.\n")

print(f"Expected SCD2 row count    : {n_master_clean + n_new + n_updates}  (updates add a new historical row)")
print(f"Actual SCD2 row count      : {len(scd2_result)}")
assert len(scd2_result) == n_master_clean + n_new + n_updates, "SCD2 row count mismatch!"
print("PASS: SCD2 row count matches expectation.")

=== ROW COUNT VALIDATION ===
Cleaned master rows        : 150
Incremental NEW customers  : 8
Incremental UPDATEs        : 20
Expected SCD1 row count    : 158  (updates overwrite, don't add rows)
Actual SCD1 row count      : 158
PASS: SCD1 row count matches expectation.

Expected SCD2 row count    : 178  (updates add a new historical row)
Actual SCD2 row count      : 178
PASS: SCD2 row count matches expectation.


In [14]:
print("=== DUPLICATE / UNIQUENESS VALIDATION ===")

dup_scd1 = scd1_result.duplicated(subset=['customer_id']).sum()
print(f"Duplicate customer_id in SCD1 table       : {dup_scd1}")
assert dup_scd1 == 0
print("PASS: SCD1 has exactly one row per customer.\n")

current_scd2 = scd2_result[scd2_result['is_current']==True]
dup_current_scd2 = current_scd2.duplicated(subset=['customer_id']).sum()
print(f"Customers with >1 'current' row in SCD2   : {dup_current_scd2}")
assert dup_current_scd2 == 0
print("PASS: SCD2 has exactly one CURRENT row per customer (older versions correctly closed).")

=== DUPLICATE / UNIQUENESS VALIDATION ===
Duplicate customer_id in SCD1 table       : 0
PASS: SCD1 has exactly one row per customer.

Customers with >1 'current' row in SCD2   : 0
PASS: SCD2 has exactly one CURRENT row per customer (older versions correctly closed).


In [15]:
print("=== COMPLETENESS VALIDATION ===")
all_incoming_ids = set(incremental_df['customer_id'])
missing_from_scd1 = all_incoming_ids - set(scd1_result['customer_id'])
missing_from_scd2 = all_incoming_ids - set(scd2_result['customer_id'])

print(f"Incoming customer_ids missing from SCD1 result: {len(missing_from_scd1)}")
print(f"Incoming customer_ids missing from SCD2 result: {len(missing_from_scd2)}")
assert len(missing_from_scd1) == 0 and len(missing_from_scd2) == 0
print("PASS: every customer from the incremental feed is reflected in both tables.")

=== COMPLETENESS VALIDATION ===
Incoming customer_ids missing from SCD1 result: 0
Incoming customer_ids missing from SCD2 result: 0
PASS: every customer from the incremental feed is reflected in both tables.


## Step 6 — Display final dataset + summary

In [16]:
print("Final SCD1 table (current state only) — sample:")
scd1_result.sample(10, random_state=1).sort_values('customer_id')

Final SCD1 table (current state only) — sample:


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
11,C-1011,Sam Farouk,Home Office,United States,New York City,New York,74895,East,2024-01-01
14,C-1014,David Nguyen,Consumer,United States,Atlanta,Georgia,46674,South,2024-06-15
19,C-1019,Liam Haidari,Home Office,United States,Minneapolis,Minnesota,25347,Central,2024-01-01
29,C-1029,David Rao,Home Office,United States,Henderson,Kentucky,62486,South,2024-01-01
40,C-1040,Tara Liu,Unknown,United States,Seattle,Washington,62518,West,2024-01-01
73,C-1073,Dana Tanaka,Consumer,United States,Chicago,Illinois,21018,Central,2024-01-01
81,C-1081,Ines Ortiz,Consumer,United States,Green Bay,Wisconsin,36446,Central,2024-01-01
95,C-1095,Hiro Sullivan,Consumer,United States,Seattle,Washington,21908,West,2024-01-01
107,C-1107,Liam Rao,Consumer,United States,Fairfield,Connecticut,92793,East,2024-06-15
124,C-1124,Mona Okafor,Consumer,United States,Minneapolis,Minnesota,57278,Central,2024-01-01


In [17]:
print("Final SCD2 table (current + historical rows) — sample of customers with history:")
customers_with_history = scd2_result[scd2_result.duplicated(subset=['customer_id'], keep=False)]
customers_with_history[['customer_id','customer_name','city','state','segment',
                          'effective_date','end_date','is_current']].head(12)

Final SCD2 table (current + historical rows) — sample of customers with history:


,customer_id,customer_name,city,state,segment,effective_date,end_date,is_current
5,C-1005,Rosa Popov,New York City,New York,Corporate,2024-01-01,2024-06-15,False
6,C-1005,Noah Kim,Columbus,Ohio,Home Office,2024-06-15,,True
8,C-1007,Sam Okafor,New York City,New York,Home Office,2024-01-01,2024-06-15,False
9,C-1007,Olga Novak,Newark,New Jersey,Home Office,2024-06-15,,True
16,C-1014,Vera Reyes,Tampa,Florida,Home Office,2024-01-01,2024-06-15,False
17,C-1014,David Nguyen,Atlanta,Georgia,Consumer,2024-06-15,,True
23,C-1020,Grace Farouk,Tampa,Florida,Consumer,2024-01-01,2024-06-15,False
24,C-1020,Zoe Ahmadi,Atlanta,Georgia,Consumer,2024-06-15,,True
53,C-1049,Alice Popov,Philadelphia,Pennsylvania,Home Office,2024-01-01,2024-06-15,False
54,C-1049,Yusuf Farouk,Monroe,Louisiana,Corporate,2024-06-15,,True


In [18]:
dt_scd1_final = DeltaTable(SILVER_SCD1_PATH)
dt_scd2_final = DeltaTable(SILVER_SCD2_PATH)

print("=== DELTA TABLE HISTORY (transaction log) — SCD1 ===")
for entry in dt_scd1_final.history():
    print(f"version {entry.get('version')}: {entry.get('operation')} | {entry.get('timestamp')}")

print("\n=== DELTA TABLE HISTORY (transaction log) — SCD2 ===")
for entry in dt_scd2_final.history():
    print(f"version {entry.get('version')}: {entry.get('operation')} | {entry.get('timestamp')}")

=== DELTA TABLE HISTORY (transaction log) — SCD1 ===
version 1: MERGE | 1785706707804
version 0: WRITE | 1785706707721

=== DELTA TABLE HISTORY (transaction log) — SCD2 ===
version 2: MERGE | 1785706707907
version 1: MERGE | 1785706707873
version 0: WRITE | 1785706707729


## Summary

- Loaded `customer_master.csv` into a Delta table (bronze layer).
- Cleaned the data: removed exact duplicates, removed duplicate `customer_id`s (keeping the
  latest), and handled nulls in `segment` and `postal_code`.
- Loaded `customer_incremental.csv` — a mix of updates to existing customers and brand-new
  customers.
- **SCD Type 1 MERGE**: overwrote changed attributes in place and inserted new customers.
  Final row count = cleaned master + new customers only (no history kept).
- **SCD Type 2 MERGE**: closed out old versions (`is_current=False`, `end_date` stamped) and
  inserted new current versions, preserving full history.
- All validation checks (row counts, duplicate checks, completeness checks) **passed**, and
  Delta's transaction log (`history()`) confirms every write as a separate, auditable version
  — the core benefit of Delta Lake over plain CSV/Parquet for incremental pipelines.
- Every `MERGE` in this notebook ran against a **real Delta table** via `deltalake` (delta-rs),
  not a simulation — the same operations, transaction log, and semantics you'd get on
  Databricks with PySpark.